# VotingClassifier: XGBoost + Random Forest

Kombiniert die besten Modelle aus den einzelnen Notebooks zu einem Ensemble.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import classification_report, f1_score
from xgboost import XGBClassifier

df = pd.read_csv('kickstarter_clean.csv')

X = df.drop(columns=['State'])
y = df['State']

RSEED = 42
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RSEED, stratify=y
)

## Modelle mit den besten Hyperparametern definieren

In [ ]:
# Bestes Random Forest Modell (aus Random_Forest_model.ipynb)
rf = RandomForestClassifier(
    max_depth=25,
    n_estimators=100,
    min_samples_split=20,
    max_features=0.3,
    random_state=RSEED,
    n_jobs=-1
)

# Bestes XGBoost Modell (aus xgboost.ipynb)
xgb = XGBClassifier(
    n_estimators=1200,
    learning_rate=0.06,
    max_depth=6,
    subsample=0.85,
    colsample_bytree=0.8,
    min_child_weight=1,
    gamma=0,
    reg_alpha=0.2,
    reg_lambda=1.2,
    scale_pos_weight=1.2,
    objective='binary:logistic',
    eval_metric='logloss',
    n_jobs=-1,
    random_state=RSEED
)

## VotingClassifier (Soft Voting)

**Soft Voting** mittelt die Wahrscheinlichkeiten beider Modelle — besser als Hard Voting, da beide Modelle `predict_proba` unterstuetzen.

In [ ]:
voting_clf = VotingClassifier(
    estimators=[
        ('random_forest', rf),
        ('xgboost', xgb)
    ],
    voting='soft'
)

voting_clf.fit(X_train, y_train)

In [ ]:
preds = voting_clf.predict(X_test)

print(classification_report(y_test, preds))
print(f"F1 weighted: {f1_score(y_test, preds, average='weighted'):.4f}")

## Vergleich: Einzelmodelle vs. VotingClassifier

In [ ]:
rf_solo = RandomForestClassifier(
    max_depth=25, n_estimators=100, min_samples_split=20,
    max_features=0.3, random_state=RSEED, n_jobs=-1
)
xgb_solo = XGBClassifier(
    n_estimators=1200, learning_rate=0.06, max_depth=6,
    subsample=0.85, colsample_bytree=0.8, min_child_weight=1,
    gamma=0, reg_alpha=0.2, reg_lambda=1.2, scale_pos_weight=1.2,
    objective='binary:logistic', eval_metric='logloss',
    n_jobs=-1, random_state=RSEED
)

rf_solo.fit(X_train, y_train)
xgb_solo.fit(X_train, y_train)

results = {
    'Random Forest':     f1_score(y_test, rf_solo.predict(X_test), average='weighted'),
    'XGBoost':           f1_score(y_test, xgb_solo.predict(X_test), average='weighted'),
    'VotingClassifier':  f1_score(y_test, preds, average='weighted'),
}

for name, score in results.items():
    print(f"{name:20s}: F1 = {score:.4f}")